# Mise en place de l'environnement GPU (Kaggle)

Prérequis, à régler une fois dans le notebook Kaggle :

- **Settings > Accelerator** : GPU T4 x2 ;
- **Settings > Internet** : activé ;
- **Add-ons > Secrets** : `HF_TOKEN`, `WANDB_API_KEY` et `GITHUB_TOKEN` (jeton GitHub
  *fine-grained*, lecture seule sur le contenu du dépôt `finetunez-votre-propre-llm`), cochés
  pour ce notebook.

Le code est cloné dans `/tmp/projet14`, hors de `/kaggle/working` : ce dossier est sauvegardé avec
le notebook, et ni le code ni les jetons ne doivent s'y retrouver. Effacez les sorties avant de
sauvegarder une version.

## 1. Secrets

In [ ]:
import os

NAMES = ("HF_TOKEN", "WANDB_API_KEY", "GITHUB_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient  # Kaggle

    secrets = UserSecretsClient()
    for name in NAMES:
        os.environ[name] = secrets.get_secret(name)
except ImportError:
    from google.colab import userdata  # Colab, en secours

    for name in NAMES:
        os.environ[name] = userdata.get(name)

os.environ["WANDB_PROJECT"] = "triage-chsa"
print({name: "ok" if os.environ.get(name) else "MANQUANT" for name in NAMES})

## 2. Code du projet

In [ ]:
import base64
import subprocess
from pathlib import Path

REPO = "https://github.com/Formation-AI-Engineer/finetunez-votre-propre-llm.git"
BRANCH = "dev"
WORKDIR = Path("/tmp/projet14")

# Le jeton passe par un en-tête HTTP : il n'est écrit ni dans l'URL ni dans .git/config.
token = base64.b64encode(f"x-access-token:{os.environ['GITHUB_TOKEN']}".encode()).decode()
auth = ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {token}"]


def git(*args, cwd=None):
    subprocess.run(["git", *auth, *args], cwd=cwd, check=True)


if (WORKDIR / ".git").exists():
    git("fetch", "--quiet", "origin", BRANCH, cwd=WORKDIR)
    git("checkout", "--quiet", BRANCH, cwd=WORKDIR)
    git("reset", "--quiet", "--hard", f"origin/{BRANCH}", cwd=WORKDIR)
else:
    git("clone", "--quiet", "--branch", BRANCH, REPO, str(WORKDIR))
os.chdir(WORKDIR)
!git log --oneline -1

## 3. Python 3.12, dépendances et contrôle du GPU

Installe uv, Python 3.12 et les versions exactes de `uv.lock`, puis lance
`scripts/check_gpu.py`. Comptez quelques minutes au premier lancement (torch et CUDA pèsent
plusieurs Gio).

In [ ]:
!bash scripts/kaggle_bootstrap.sh